# Garmin Connect → Local SQLite

Downloads all Garmin Connect data (activities, monitoring, sleep, HR, HRV, weight) and imports it into local SQLite databases under `data/sqlite/`.

**Prerequisites:**
- `garmindb` installed (`uv pip install garmindb`)
- Garmin Connect account credentials

**What this does:**
1. Configures `GarminConnectConfig.json` with your credentials and local storage paths
2. Runs `garmindb_cli.py --all --download --import --analyze` to download everything
3. Shows the resulting SQLite databases

## 1. Setup Configuration

Enter your Garmin Connect credentials. The config file is stored at `~/.GarminDb/GarminConnectConfig.json` and databases go to `data/sqlite/`.

In [8]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(repo_root))

import importlib
import garmin
import garmin.config
import garmin.cli_wrapper
importlib.reload(garmin.config)
importlib.reload(garmin.cli_wrapper)
importlib.reload(garmin)

from garmin import setup_config, get_config_path, get_db_dir

print(f"Config path: {get_config_path()}")
print(f"Database dir: {get_db_dir()}")

if get_config_path().exists():
    print("\n✅ Config already exists. Skip to Step 2, or re-run this cell to update.")
else:
    print("\n⚠️  No config found. Enter your credentials below.")

Config path: /Users/tonkata/.GarminDb/GarminConnectConfig.json
Database dir: /Users/tonkata/repos/workoutdata/data/sqlite

✅ Config already exists. Skip to Step 2, or re-run this cell to update.


In [9]:
import getpass

username = input("Garmin Connect email: ")
password = getpass.getpass("Garmin Connect password: ")

setup_config(
    username,
    password,
    monitoring_start_date="11/01/2025",
    sleep_start_date="11/01/2025",
    weight_start_date="11/01/2025",
    rhr_start_date="11/01/2025",
    hrv_start_date="11/01/2025",
    download_all_activities=1000,
)

✅ GarminDB config written to: /Users/tonkata/.GarminDb/GarminConnectConfig.json
   Database directory: /Users/tonkata/repos/workoutdata/data/sqlite


PosixPath('/Users/tonkata/.GarminDb/GarminConnectConfig.json')

## 2. Download All Data

This downloads **all** data from Garmin Connect, imports it into SQLite, and runs analysis to create summary tables. This can take a while on the first run depending on how much history you have.

In [10]:
from garmin import download_all

result = download_all()
print(f"\n✅ Done (exit code: {result.returncode})")


GARMIN: Downloading ALL data from Garmin Connect...
🔄 Running: /Users/tonkata/repos/workoutdata/.venv/bin/python /Users/tonkata/repos/workoutdata/.venv/bin/garmindb_cli.py -f /Users/tonkata/.GarminDb --all --download --import --analyze
___Downloading All Data___
Getting activities: '/Users/tonkata/repos/workoutdata/data/sqlite/FitFiles/Activities' (1000) temp /var/folders/vn/xgnj5jk149x4rcfj81h6n5zr0000gn/T/tmp_l6p1ggv
___Importing All Data___
Processing user settings data
Processing profile data
Processing profile data: {'measurement_system': 'DisplayMeasure.metric', 'gender': 'Gender.male', 'weight': 81.646, 'height': 1.75, 'vo2max_running': None, 'vo2max_cycling': None, 'handedness': 'right'}
Processing user personal information data
Processing profile data
Processing profile data: {'locale': 'en', 'time_zone': 'America/Los_Angeles', 'country_code': 'US'}
Processing user settings data
Processing profile data
Processing profile data: {'id': 433453919, 'userName': 'sinnerson@gmail.co


100%|██████████| 193/193 [00:03<00:00, 51.39activities/s]

100%|██████████| 135/135 [03:10<00:00,  1.41s/days]

100%|██████████| 135/135 [02:35<00:00,  1.15s/days]

100%|██████████| 135/135 [04:09<00:00,  1.85s/days]

100%|██████████| 135/135 [02:40<00:00,  1.19s/days]

100%|██████████| 135/135 [02:40<00:00,  1.19s/days]

100%|██████████| 135/135 [02:29<00:00,  1.11s/days]

100%|██████████| 135/135 [02:29<00:00,  1.11s/days]

100%|██████████| 1/1 [00:00<00:00, 96.09files/s]

100%|██████████| 1/1 [00:00<00:00, 303.41files/s]

100%|██████████| 1/1 [00:00<00:00, 246.19files/s]

100%|██████████| 135/135 [00:00<00:00, 6789.67files/s]

100%|██████████| 805/805 [00:00<00:00, 839.06files/s]

100%|██████████| 727/727 [00:00<00:00, 984.72files/s] 

100%|██████████| 2155/2155 [02:54<00:00, 12.37files/s]

100%|██████████| 135/135 [00:05<00:00, 23.32files/s]

100%|██████████| 135/135 [00:00<00:00, 961.23files/s]

100%|██████████| 135/135 [00:00<00:00, 1068.09files/s]

100%|██████████| 193/193 [00:

## 3. Verify SQLite Databases

Check the downloaded databases and their sizes.

In [5]:
from garmin import get_db_dir

db_dir = get_db_dir()
print(f"Database directory: {db_dir}\n")

db_files = sorted(db_dir.rglob("*.db"))
if db_files:
    for db in db_files:
        size_mb = db.stat().st_size / (1024 * 1024)
        print(f"  📦 {db.relative_to(db_dir)}  ({size_mb:.2f} MB)")
else:
    print("  ⚠️  No .db files found. The download may have failed.")

Database directory: /Users/tonkata/repos/workoutdata/data/sqlite

  📦 DBs/garmin_monitoring.db  (0.09 MB)


## 4. Quick Preview

Peek at the tables in each database.

In [12]:
import sqlite3

db_dir = get_db_dir()

for db_path in sorted(db_dir.rglob("*.db")):
    print(f"\n{'='*60}")
    print(f"📦 {db_path.relative_to(db_dir)}")
    print(f"{'='*60}")
    try:
        conn = sqlite3.connect(str(db_path))
        tables = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
        for (table,) in tables:
            count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]
            print(f"  {table}: {count:,} rows")
        conn.close()
    except Exception as e:
        print(f"  ⚠️  Error reading: {e}")


📦 DBs/garmin.db
  _attributes: 23 rows
  attributes: 25 rows
  daily_summary: 805 rows
  device_info: 3,826 rows
  devices: 7 rows
  files: 1,225 rows
  hrv: 109 rows
  resting_hr: 128 rows
  sleep: 135 rows
  sleep_events: 1,939 rows
  stress: 179,661 rows
  weight: 1 rows

📦 DBs/garmin_activities.db
  _attributes: 14 rows
  activities: 193 rows
  activities_devices: 1,269 rows
  activity_laps: 224 rows
  activity_records: 80,295 rows
  activity_splits: 0 rows
  climbing_activities: 0 rows
  cycle_activities: 24 rows
  paddle_activities: 0 rows
  steps_activities: 2 rows

📦 DBs/garmin_monitoring.db
  _attributes: 10 rows
  monitoring: 43,924 rows
  monitoring_climb: 3,488 rows
  monitoring_hr: 146,714 rows
  monitoring_hrv_status: 164 rows
  monitoring_hrv_value: 10,355 rows
  monitoring_info: 1,736 rows
  monitoring_intensity: 1,062 rows
  monitoring_pulse_ox: 41,241 rows
  monitoring_rr: 134,289 rows
  sqlite_stat1: 1 rows
  sqlite_stat4: 10 rows

📦 DBs/garmin_summary.db
  _attribu

## 5. Sleep Overview

Query the `sleep` table from `garmin.db` and join with HRV data from `garmin_monitoring.db` to show a daily sleep report.

In [11]:
import sqlite3
import pandas as pd
from garmin import get_db_dir

db_dir = get_db_dir()
garmin_db = db_dir / "DBs" / "garmin.db"
monitoring_db = db_dir / "DBs" / "garmin_monitoring.db"

conn = sqlite3.connect(str(garmin_db))
sleep_df = pd.read_sql_query(
    "SELECT day, total_sleep, score, qualifier FROM sleep ORDER BY day DESC",
    conn,
)
conn.close()

conn = sqlite3.connect(str(monitoring_db))
hrv_df = pd.read_sql_query(
    "SELECT timestamp, last_night AS hrv FROM monitoring_hrv_status ORDER BY timestamp DESC",
    conn,
)
conn.close()

sleep_df["day"] = pd.to_datetime(sleep_df["day"])
hrv_df["day"] = pd.to_datetime(hrv_df["timestamp"]).dt.normalize()
hrv_df = hrv_df.drop(columns=["timestamp"])

def time_str_to_hours(t):
    if pd.isna(t) or t == "00:00:00":
        return None
    parts = str(t).split(":")
    return int(parts[0]) + int(parts[1]) / 60 + int(parts[2]) / 3600

sleep_df["hours"] = sleep_df["total_sleep"].apply(time_str_to_hours)

df = sleep_df.merge(hrv_df, on="day", how="left")
df = df[["day", "hours", "hrv", "score", "qualifier"]]
df.columns = ["Day", "Sleep (hrs)", "HRV (ms)", "Score", "Qualifier"]
df["Day"] = df["Day"].dt.strftime("%Y-%m-%d")
df["Sleep (hrs)"] = df["Sleep (hrs)"].round(1)

print(f"Total sleep records: {len(df)}")
df.head(30)

ValueError: invalid literal for int() with base 10: '00.000000'

---

### Incremental Updates

After the initial full download, use `download_latest()` to sync only recent data:

In [ ]:
# from garmin import download_latest
# result = download_latest()
# print(f"\n✅ Done (exit code: {result.returncode})")